In [1]:
import torch
import pandas as pd
import re
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from tqdm.auto import tqdm


In [2]:
torch.cuda.empty_cache()

In [3]:
base_model = "mistralai/Mistral-7B-v0.3"
lora_path = "mistral_mcq_sft_model"

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    llm_int8_enable_fp32_cpu_offload=True
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map={"": 0},   
    trust_remote_code=True
)


model = PeftModel.from_pretrained(model, lora_path)

model.eval()


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): lora.Linear4bit(
                (base_layer): Linea

In [4]:
def build_prompt(row):
    return f"""You are an expert MCQ solver.

Question: {row['question']}

Options:
a) {row['option_a']}
b) {row['option_b']}
c) {row['option_c']}
d) {row['option_d']}

INSTRUCTIONS:
- Choose exactly one: a, b, c, d
- If uncertain → idk
- Always provide reasoning

OUTPUT FORMAT (STRICT):
<answer>a/b/c/d/idk</answer>
<reasoning>2-3 line explanation</reasoning>
"""

In [5]:
def parse_output(output):
    final_answer = "idk"
    reasoning = ""

    ans_match = re.search(r"<answer>\s*(.*?)\s*</answer>", output, re.IGNORECASE)
    if ans_match:
        final_answer = ans_match.group(1).strip().lower()

    reason_match = re.search(r"<reasoning>\s*(.*?)\s*</reasoning>", output, re.IGNORECASE | re.DOTALL)
    if reason_match:
        reasoning = reason_match.group(1).strip()

    if final_answer not in ["a", "b", "c", "d", "idk"]:
        final_answer = "idk"

    if reasoning == "":
        final_answer = "idk"

    return final_answer, reasoning


In [6]:
input_file = "final_combined_dataset.csv"
output_file = "abstention_sft_mistral_final.csv"
save_every = 50
df = pd.read_csv(input_file)

if os.path.exists(output_file):
    done = pd.read_csv(output_file)
    start_idx = len(done)
    print(f"Resuming from {start_idx}")
else:
    start_idx = 0
    done = pd.DataFrame()

In [ ]:
batch_size = 8

results = []

for start in tqdm(range(start_idx, len(df), batch_size), desc="Inference"):

    batch = df.iloc[start:start + batch_size]

    prompts = [build_prompt(row) for _, row in batch.iterrows()]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for i, text in enumerate(decoded):
        prompt = prompts[i]
        text = text[len(prompt):]

        ans, reason = parse_output(text)

        row = batch.iloc[i]

        results.append({
            "item_id": row["item_id"],
            "question": row["question"],
            "final_answer": row["final_answer"],
            "option_a": row["option_a"],
            "option_b": row["option_b"],
            "option_c": row["option_c"],
            "option_d": row["option_d"],
            "question_level": row["question_level"],
            "language": row["language"],
            "reasoning": row["reasoning"],
            "pred_final_answer": ans,
            "pred_reasoning": reason
        })

    if len(results) >= 50:
        temp_df = pd.DataFrame(results)

        if os.path.exists(output_file):
            temp_df.to_csv(output_file, mode="a", index=False, header=False)
        else:
            temp_df.to_csv(output_file, index=False)

        print(f"💾 Saved batch up to {start + batch_size}")
        results = []

Inference:   0%|          | 0/441 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


💾 Saved batch up to 56


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


💾 Saved batch up to 112


...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


💾 Saved batch up to 3080


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


💾 Saved batch up to 3136


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [ ]:
if len(results) > 0:
    temp_df = pd.DataFrame(results)

    if os.path.exists(output_file):
        temp_df.to_csv(output_file, mode="a", index=False, header=False)
    else:
        temp_df.to_csv(output_file, index=False)

print("Inference complete with checkpoint saving!")